# Heart Disease Cloud Training

This notebook is designed for Google Colab's hosted CPU runtime. It validates the Cleveland dataset, performs leakage-free cross-validation with complete preprocessing-plus-model pipelines, evaluates one selected pipeline on an untouched test set, exports a separate cloud release package, and verifies model reload. It does not modify the existing local artifacts.

## Reproducibility and safety

- Upload `processed.cleveland.data` when prompted; no Windows paths are used.
- Do not upload real patient records.
- Duplicate rows are reported, not silently deleted.
- Missing values are handled inside fitted pipelines; they are never replaced with zero.
- The held-out test set is not used for model selection or tuning.
- Results are research measurements on a small dataset, not clinical validation or medical diagnosis.

In [ ]:
# Colab setup and reproducibility
import json
import platform
import shutil
import sys
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from google.colab import files
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, brier_score_loss, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_SEED = 42
EXPECTED_ROWS = 303
EXPECTED_COLUMNS = 14
COLUMNS = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']
FEATURE_NAMES = COLUMNS[:-1]
NUMERICAL_FEATURES = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
CATEGORICAL_FEATURES = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
EXPECTED_CATEGORIES = {'sex': {0, 1}, 'cp': {1, 2, 3, 4}, 'fbs': {0, 1}, 'restecg': {0, 1, 2}, 'exang': {0, 1}, 'slope': {1, 2, 3}, 'ca': {0, 1, 2, 3}, 'thal': {3, 6, 7}}
OUTPUT_DIR = Path('heart_disease_cloud_release')
OUTPUT_DIR.mkdir(exist_ok=True)
print('Python:', sys.version)
print('Platform:', platform.platform())
print('Pandas:', pd.__version__)
print('Scikit-learn:', sklearn.__version__)
print('Random seed:', RANDOM_SEED)

## A. Upload and validate the dataset

In [ ]:
uploaded = files.upload()
if 'processed.cleveland.data' not in uploaded:
    raise ValueError('Upload a file named processed.cleveland.data.')
dataset_path = Path('processed.cleveland.data')
raw_df = pd.read_csv(dataset_path, header=None, names=COLUMNS, na_values='?', skipinitialspace=True)
if raw_df.shape != (EXPECTED_ROWS, EXPECTED_COLUMNS):
    raise ValueError(f'Expected {(EXPECTED_ROWS, EXPECTED_COLUMNS)}, received {raw_df.shape}.')
if list(raw_df.columns) != COLUMNS:
    raise ValueError('Dataset columns do not match the expected Cleveland schema.')
print('Shape:', raw_df.shape)
display(raw_df.head())
display(raw_df.dtypes.rename('dtype').to_frame())
display(raw_df.isna().sum().rename('missing_count').to_frame())
duplicate_count = int(raw_df.duplicated().sum())
print('Duplicate rows:', duplicate_count)
print('Duplicate handling: duplicates are retained and reported; they are not silently deleted.')
print('Original target values:', sorted(raw_df['num'].dropna().unique().tolist()))
if raw_df['num'].isna().any():
    raise ValueError('Target contains missing values; stop rather than imputing the target.')
if set(raw_df['num'].unique()) != {0, 1, 2, 3, 4}:
    raise ValueError('Target contains unexpected values.')
for feature, allowed in EXPECTED_CATEGORIES.items():
    observed = set(raw_df[feature].dropna().unique())
    unexpected = observed - allowed
    if unexpected:
        raise ValueError(f'{feature} contains unexpected values: {unexpected}')
    print(feature, 'observed:', sorted(observed), 'expected:', sorted(allowed))
y_binary = (raw_df['num'] > 0).astype(int)
X = raw_df[FEATURE_NAMES].copy()
print('Binary target mapping: original 0 -> 0; original 1,2,3,4 -> 1')
display(y_binary.value_counts().sort_index().rename('count').to_frame())

## B. Exploratory data analysis
Charts describe this dataset only. Correlation is association, not causation.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.countplot(data=raw_df, x='num', ax=axes[0]); axes[0].set_title('Original target distribution'); axes[0].set_xlabel('Original num')
sns.countplot(x=y_binary, ax=axes[1]); axes[1].set_title('Binary target distribution'); axes[1].set_xlabel('Disease-present dataset label')
sns.histplot(raw_df['age'], bins=20, kde=True, ax=axes[2]); axes[2].set_title('Age distribution')
plt.tight_layout(); plt.show()

raw_df[NUMERICAL_FEATURES].hist(figsize=(15, 10), bins=20, layout=(2, 3))
plt.suptitle('Numerical feature distributions'); plt.tight_layout(); plt.show()

plt.figure(figsize=(12, 8))
sns.heatmap(raw_df[FEATURE_NAMES + ['num']].corr(numeric_only=True), cmap='coolwarm', center=0, annot=False)
plt.title('Clinical feature correlation heatmap'); plt.tight_layout(); plt.show()

plt.figure(figsize=(12, 4)); raw_df.isna().sum().plot.bar(); plt.title('Missing-value analysis'); plt.ylabel('Missing observations'); plt.tight_layout(); plt.show()


## C. Leakage-free preprocessing and split
The test set remains untouched until final evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y_binary, test_size=0.20, stratify=y_binary, random_state=RANDOM_SEED)
def make_preprocessor():
    numeric = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
    categorical = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))])
    return ColumnTransformer([('numeric', numeric, NUMERICAL_FEATURES), ('categorical', categorical, CATEGORICAL_FEATURES)])
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_SEED),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_SEED),
    'Random Forest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1),
}
print('Training shape:', X_train.shape, 'Testing shape:', X_test.shape)
display(pd.DataFrame({'train': y_train.value_counts().sort_index(), 'test': y_test.value_counts().sort_index()}))
print('Preprocessing is created inside every complete model pipeline and is therefore fitted separately in every CV fold.')

## D. Leakage-free cross-validation
ROC-AUC is the predefined selection criterion; all requested metrics are reported.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
scoring = {'accuracy': 'accuracy', 'precision': 'precision', 'recall': 'recall', 'f1': 'f1', 'roc_auc': 'roc_auc'}
cv_rows = []
cv_pipelines = {}
for name, classifier in models.items():
    pipeline = Pipeline([('preprocessor', make_preprocessor()), ('classifier', clone(classifier))])
    cv_pipelines[name] = pipeline
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, return_train_score=False, n_jobs=-1)
    row = {'model': name}
    for metric in scoring:
        values = scores[f'test_{metric}']
        row[f'{metric}_folds'] = values.tolist(); row[f'{metric}_mean'] = float(values.mean()); row[f'{metric}_std'] = float(values.std())
    cv_rows.append(row)
    print(name)
    for metric in scoring: print(f"  {metric}: {np.round(scores[f'test_{metric}'], 4)} | mean={row[f'{metric}_mean']:.4f} | std={row[f'{metric}_std']:.4f}")
cv_results = pd.DataFrame(cv_rows).sort_values('roc_auc_mean', ascending=False).reset_index(drop=True)
selected_model_name = cv_results.loc[0, 'model']
display(cv_results[['model'] + [f'{m}_{s}' for m in scoring for s in ['mean', 'std']]])
print('Selected by mean CV ROC-AUC:', selected_model_name)

## E. Final training and F. held-out test evaluation

In [ ]:
final_pipeline = cv_pipelines[selected_model_name].fit(X_train, y_train)
y_pred = final_pipeline.predict(X_test)
classes = list(final_pipeline.named_steps['classifier'].classes_)
y_probability = None
if hasattr(final_pipeline, 'predict_proba'):
    probabilities = final_pipeline.predict_proba(X_test)
    y_probability = probabilities[:, classes.index(1)]
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
test_metrics = {'accuracy': float(accuracy_score(y_test, y_pred)), 'precision': float(precision_score(y_test, y_pred, zero_division=0)), 'recall_sensitivity': float(recall_score(y_test, y_pred, zero_division=0)), 'specificity': float(tn / (tn + fp)) if (tn + fp) else None, 'f1': float(f1_score(y_test, y_pred, zero_division=0)), 'roc_auc': float(roc_auc_score(y_test, y_probability)) if y_probability is not None else None, 'brier_score': float(brier_score_loss(y_test, y_probability)) if y_probability is not None else None, 'test_observations': int(len(y_test)), 'class_support': {str(k): int(v) for k, v in y_test.value_counts().sort_index().items()}}
print(json.dumps(test_metrics, indent=2)); print(classification_report(y_test, y_pred, target_names=['No disease label', 'Disease label'], zero_division=0))
pd.DataFrame(cm, index=['actual_0', 'actual_1'], columns=['predicted_0', 'predicted_1']).to_csv(OUTPUT_DIR / 'confusion_matrix.csv')
plt.figure(figsize=(5, 4)); sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No disease', 'Disease'], yticklabels=['No disease', 'Disease']); plt.xlabel('Predicted label'); plt.ylabel('Actual label'); plt.title('Confusion matrix'); plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=200); plt.show()
if y_probability is not None:
    fpr, tpr, _ = roc_curve(y_test, y_probability)
    plt.figure(figsize=(6, 5)); plt.plot(fpr, tpr, label=f"ROC-AUC={test_metrics['roc_auc']:.3f}"); plt.plot([0, 1], [0, 1], '--', color='gray'); plt.xlabel('False positive rate'); plt.ylabel('True positive rate'); plt.title('ROC curve'); plt.legend(); plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'roc_curve.png', dpi=200); plt.show()
with open(OUTPUT_DIR / 'classification_report.txt', 'w') as f: f.write(classification_report(y_test, y_pred, target_names=['No disease label', 'Disease label'], zero_division=0))

In [ ]:
comparison_plot = cv_results.set_index('model')[[f'{m}_mean' for m in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']]]
comparison_plot.columns = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
comparison_plot.plot(kind='bar', figsize=(12, 6), ylim=(0, 1), title='Cross-validation model comparison'); plt.ylabel('Mean score'); plt.xticks(rotation=0); plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'model_comparison.png', dpi=200); plt.show()
metric_names = ['accuracy', 'precision', 'recall_sensitivity', 'specificity', 'f1', 'roc_auc', 'brier_score']
metric_frame = pd.DataFrame({'metric': metric_names, 'value': [test_metrics[name] for name in metric_names]}).dropna()
metric_frame.plot(x='metric', y='value', kind='bar', figsize=(10, 5), legend=False, title='Selected model test metrics'); plt.ylim(0, 1); plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'classification_metrics_comparison.png', dpi=200); plt.show()
if hasattr(final_pipeline.named_steps['classifier'], 'predict_proba'):
    permutation = permutation_importance(final_pipeline, X_test, y_test, scoring='roc_auc', n_repeats=20, random_state=RANDOM_SEED, n_jobs=-1)
    importance = pd.Series(permutation.importances_mean, index=FEATURE_NAMES).sort_values(ascending=False)
    importance.to_csv(OUTPUT_DIR / 'permutation_importance.csv', header=['mean_importance'])
    importance.sort_values().plot.barh(figsize=(8, 5), title='Permutation importance on held-out test data'); plt.xlabel('Mean ROC-AUC decrease'); plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'permutation_importance.png', dpi=200); plt.show()
print('Permutation importance indicates predictive contribution only; it does not establish causation or clinical significance.')

## G. Export a separate cloud artifact package
The combined pipeline is intentionally exported separately from the existing local artifacts. It contains fitted preprocessing and classifier steps.

In [ ]:
pipeline_path = OUTPUT_DIR / 'heart_disease_pipeline.joblib'
joblib.dump(final_pipeline, pipeline_path)
metadata = {'project_name': 'AI-Based Early Prediction of Heart Disease Using Clinical Data Analytics', 'dataset_source': 'UCI Heart Disease processed Cleveland dataset', 'dataset_filename': 'processed.cleveland.data', 'feature_names': FEATURE_NAMES, 'target_column': 'num', 'binary_target_mapping': {'0': 0, '1,2,3,4': 1}, 'numerical_features': NUMERICAL_FEATURES, 'categorical_features': CATEGORICAL_FEATURES, 'imputation': {'numeric': 'median', 'categorical': 'most_frequent'}, 'scaling': 'StandardScaler', 'encoding': 'OneHotEncoder(handle_unknown="ignore")', 'train_test_split': {'test_size': 0.2, 'stratify': True, 'random_state': RANDOM_SEED}, 'cross_validation': {'type': 'StratifiedKFold', 'n_splits': 5, 'shuffle': True, 'random_state': RANDOM_SEED, 'selection_metric': 'mean ROC-AUC'}, 'selected_model_name': selected_model_name, 'final_test_metrics': test_metrics, 'python_version': sys.version, 'scikit_learn_version': sklearn.__version__, 'training_timestamp_utc': datetime.now(timezone.utc).isoformat(), 'model_classes': classes, 'processed_feature_count': len(final_pipeline.named_steps['preprocessor'].get_feature_names_out()), 'probability_warning': 'Class-1 probability is a model estimate for the dataset label, not a calibrated clinical risk.', 'limitation': 'Academic research prototype; not clinically validated and not a medical diagnostic system.'}
(OUTPUT_DIR / 'model_metadata.json').write_text(json.dumps(metadata, indent=2, default=str))
(OUTPUT_DIR / 'evaluation_metrics.json').write_text(json.dumps(test_metrics, indent=2))
cv_results.to_csv(OUTPUT_DIR / 'cross_validation_results.csv', index=False)
archive_path = shutil.make_archive('heart_disease_cloud_release', 'zip', root_dir=OUTPUT_DIR)
print('Exported:', sorted(p.name for p in OUTPUT_DIR.iterdir()))
print('ZIP:', archive_path)

## H. Reload and round-trip inference verification
One held-out row is used only as an inference demonstration. The round-trip prediction must match.

In [ ]:
reloaded_pipeline = joblib.load(pipeline_path)
reloaded_preprocessor = reloaded_pipeline.named_steps['preprocessor']
reloaded_classifier = reloaded_pipeline.named_steps['classifier']
if list(reloaded_preprocessor.feature_names_in_) != FEATURE_NAMES: raise AssertionError('Reloaded feature names do not match.')
if len(reloaded_preprocessor.get_feature_names_out()) != metadata['processed_feature_count']: raise AssertionError('Reloaded feature count does not match.')
if list(reloaded_classifier.classes_) != classes: raise AssertionError('Reloaded classes do not match.')
held_out_row = X_test.iloc[[0]]
original_prediction = int(final_pipeline.predict(held_out_row)[0])
reloaded_prediction = int(reloaded_pipeline.predict(held_out_row)[0])
if original_prediction != reloaded_prediction: raise AssertionError('Reloaded prediction differs from fitted prediction.')
if hasattr(reloaded_pipeline, 'predict_proba'): print('Reloaded probability:', reloaded_pipeline.predict_proba(held_out_row)[0].tolist())
print('Round-trip verification passed:', {'prediction': reloaded_prediction, 'raw_feature_count': len(FEATURE_NAMES), 'processed_feature_count': metadata['processed_feature_count'], 'classes': classes})

In [ ]:
files.download('heart_disease_cloud_release.zip')
print('Upload heart_disease_cloud_release.zip to the local project only after reviewing the metrics and round-trip verification. The existing backend currently remains on its original separate artifacts.')